In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

: 

In [ ]:
df= pd.read_csv("train.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(x='Survived',data=df,palette='hls')
plt.title('Survived vs Not Survived')
plt.xticks(ticks=[0,1],labels=['Not Survived','Survived'])
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
plt.pie(df['Survived'].value_counts(),labels=['None Survived','Survived'],autopct='%1.1f%%',colors=['#ff9999','#66b3ff'])
plt.title('Survived vs Not Survived')

plt.tight_layout()
plt.show()

In [ ]:
df['Pclass'].value_counts()

In [ ]:
sns.countplot(x='Pclass',data=df,palette='hls', order=df['Pclass'].value_counts().index,hue='Survived')
plt.title('Passenger Class Distribution')
plt.xlabel('Passenger Class')
plt.xticks(ticks=[0,1,2],labels=['Class 1','Class 2','Class 3'])
plt.legend(title='Survived', loc='upper right', labels=['Not Survived','Survived'])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.title("Passenger Class Distribution in Titanic")
plt.pie(df['Pclass'].value_counts(),labels=['class 1','class 2','class 3'],autopct='%1.1f%%',colors=['#ff9999','#66b3ff',"#354a5f"])
plt.tight_layout()
plt.show()

In [ ]:
sns.countplot(x='Sex',data=df,palette='hls',hue='Survived')
plt.title("Gender Distribution in titanic")
plt.legend(title='Survived',labels=['Not Survived','Survived'])
plt.tight_layout()
plt.show()

In [ ]:
plt.pie(data=df, x=df['Sex'].value_counts(),autopct="%1.1f%%", colors=['#ff9999','#66b3ff'],labels=['male','female'])
plt.title("Sex Distribution - pie chart")
plt.tight_layout()
plt.show()

In [ ]:

df["Embarked"] = df["Embarked"].map({
    "S": "Southampton",
    "C": "Cherbourg",
    "Q": "Queenstown"
})

In [ ]:
sns.countplot(x='Embarked',data=df,palette='hls',hue='Survived')
plt.legend(labels=['Not Survived','Survived'],title='Survived')
plt.title("Embarked Distribution in titanic")
plt.tight_layout()

In [ ]:
plt.pie(data=df, x=df['Embarked'].value_counts(),autopct="%1.1f%%", colors=['#ff9999','#66b3ff','#354a5f'],labels=['Southampton','Cherbourg ','Queenstown '])
plt.title("Embarked Port Distribution") 
plt.tight_layout()
plt.show()

In [ ]:
sns.countplot(x='Parch',data=df,palette='hls',hue='Survived')

In [ ]:
sns.histplot(data=df,x='Age',bins=30,kde=True,color="#df9beb",fill=True)

In [ ]:
sns.kdeplot(data=df,x='Age',hue='Survived',fill=True,palette='hls')

In [ ]:
sns.barplot(data=df,x='Parch',y='SibSp',hue='Survived',palette='hls')
plt.title("Survival Distribution by Number of Parents/Children and Siblings/Spouses")
plt.tight_layout()

In [ ]:
corr_matrix = df.corr(numeric_only=True)
sns.heatmap(corr_matrix,annot=True,cmap='coolwarm',fmt='.2f')
plt.title("Correlation Matrix of Titanic Dataset")
plt.tight_layout()

In [ ]:
X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]
y = df['Survived']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2,stratify=y)

In [ ]:
print(X_train.isnull().sum())


In [ ]:
print(y_train.isnull().sum())

In [ ]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy="median")

# learn median from TRAIN only
X_train[["Age"]] = num_imputer.fit_transform(X_train[["Age"]])

# reuse the TRAIN median on test (no fit here!)
X_test[["Age"]] = num_imputer.transform(X_test[["Age"]])

print(num_imputer.statistics_)   # the median it learned from train

In [ ]:
print(X_train.isnull().sum())

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

num_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
cat_cols = ['Sex', 'Embarked']

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),  # adds an "Age was missing" flag
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="if_binary")),
])

preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])

model = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=1000)),
])
DT_model = Pipeline([
    ("preprocess", preprocess),
    ("clf", DecisionTreeClassifier(max_depth=100)),
])
    

model.fit(X_train, y_train)          # imputer, scaler and encoder all learn from train only
print("LogisticRegression Test accuracy:", model.score(X_test, y_test))
DT_model.fit(X_train, y_train)          # imputer, scaler and encoder all learn from train only
print("Decision tree Test accuracy:", DT_model.score(X_test, y_test))

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X_train, y_train, cv=5)
print(scores.mean(), scores.std())